In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

#  Source data
df = spark.read.format("delta").table("sales.silver.products")

In [0]:
#  Add HASH column 
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("ProductID").cast("string"),
        col("ProductName"),
        col("Category"),
        col("Price").cast("string"),
        col("ModifiedDate").cast("string")
    ))
)

In [0]:
# Target table
table_name = "sales.gold.dim_product"
delta_tgt = DeltaTable.forName(spark, table_name)


In [0]:

# MERGE (SCD TYPE 1)
(
    delta_tgt.alias("tgt")
    .merge(
        df_hash.alias("src"),
        "tgt.ProductID = src.ProductID"
    )
    .whenMatchedUpdate(
        condition = "tgt.HASHVALUE != src.src_hash",
        set = {
            "ProductID": "src.ProductID",
            "ProductName": "src.ProductName",
            "Category": "src.Category",
            "Price": "src.Price",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks-updated")
        }
    )
    .whenNotMatchedInsert(
        values = {
            "ProductID": "src.ProductID",
            "ProductName": "src.ProductName",
            "Category": "src.Category",
            "Price": "src.Price",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks")
        }
    )
    .execute()
)